# Nombre de PR fusionnées par personne agrégées par semaine

Ce notebook récupère, via l'API GitHub, le nombre de *pull requests* (PR) fusionnées
pour **un ou plusieurs dépôts**, les regroupe par auteur et par semaine sur l'année écoulée,
puis affiche le résultat sous forme de graphique.

**Dépendances :** `requests`, `pandas`, `matplotlib`.

**Token GitHub :** l'API GitHub limite les appels non authentifiés à 60 requêtes par heure.
Pour lever cette limite, définissez la variable d'environnement `GITHUB_TOKEN`
avec un *Personal Access Token* (PAT) GitHub :

```bash
export GITHUB_TOKEN=ghp_xxxxxxxxxxxxxxxxxxxx
```

Sans token, le notebook fonctionne mais peut être limité sur de grands dépôts.

In [ ]:
import os
import datetime
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

## Paramètres

Modifiez `REPOS` pour lister les dépôts à analyser sous la forme
`[(owner, repo), ...]`. Vous pouvez ajouter autant de dépôts que vous le souhaitez.

In [ ]:
REPOS = [
    ("sdpython", "teachpyx"),
    # ("sdpython", "onnx-extended"),  # ajoutez d'autres dépôts ici
]

# Jeton d'authentification GitHub (optionnel mais recommandé)
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "")

## Récupération des PR fusionnées via l'API GitHub

L'API REST GitHub expose le point d'accès `/repos/{owner}/{repo}/pulls`
avec `state=closed`. On filtre ensuite les PR dont le champ `merged_at` est renseigné
et dont la date de fusion est dans les 12 derniers mois.

La pagination est gérée via le paramètre `page`.
La boucle principale itère sur chaque dépôt listé dans `REPOS`.

In [ ]:
def fetch_merged_prs(owner: str, repo: str, token: str = "") -> list[dict]:
    """Récupère toutes les PR fusionnées au cours de l'année écoulée pour un dépôt.

    :param owner: propriétaire du dépôt GitHub
    :param repo: nom du dépôt GitHub
    :param token: jeton d'authentification GitHub (optionnel)
    :return: liste de dictionnaires avec les champs ``author``, ``merged_at``, ``repo``
    """
    headers = {"Accept": "application/vnd.github+json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    since = datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(days=365)

    results = []
    page = 1
    per_page = 100

    while True:
        url = (
            f"https://api.github.com/repos/{owner}/{repo}/pulls"
            f"?state=closed&per_page={per_page}&page={page}&sort=updated&direction=desc"
        )
        response = requests.get(url, headers=headers, timeout=30)
        try:
            response.raise_for_status()
        except requests.HTTPError as exc:
            status = exc.response.status_code
            if status == 401:
                raise RuntimeError(
                    "Authentification refusée (401). Vérifiez votre GITHUB_TOKEN."
                ) from exc
            if status == 403:
                raise RuntimeError(
                    "Accès refusé (403). Vous avez peut-être atteint la limite de l'API "
                    "GitHub (60 requêtes/h sans token). Définissez GITHUB_TOKEN."
                ) from exc
            if status == 404:
                raise RuntimeError(
                    f"Dépôt introuvable (404) : {owner}/{repo}. Vérifiez OWNER et REPO."
                ) from exc
            raise
        prs = response.json()

        if not prs:
            break

        stop = False
        for pr in prs:
            merged_at = pr.get("merged_at")
            if not merged_at:
                continue
            merged_dt = datetime.datetime.fromisoformat(merged_at.replace("Z", "+00:00"))
            if merged_dt < since:
                stop = True
                break
            author = (pr.get("user") or {}).get("login", "unknown")
            results.append({"author": author, "merged_at": merged_dt, "repo": f"{owner}/{repo}"})

        if stop:
            break

        page += 1

    return results


merged_prs = []
for owner, repo in REPOS:
    prs = fetch_merged_prs(owner, repo, GITHUB_TOKEN)
    print(f"  {owner}/{repo} : {len(prs)} PR(s) fusionnée(s)")
    merged_prs.extend(prs)

print(f"Total : {len(merged_prs)} PR(s) fusionnée(s) sur l'ensemble des dépôts.")

## Agrégation par auteur et par semaine

In [ ]:
df = pd.DataFrame(merged_prs)

if df.empty:
    print("Aucune donnée à afficher.")
else:
    # Tronque la date au lundi de la semaine
    df["week"] = df["merged_at"].dt.to_period("W").dt.start_time

    weekly = (
        df.groupby(["repo", "author", "week"])
        .size()
        .reset_index(name="pr_count")
    )
    print(weekly.head(10))

## Tableau croisé (auteur × semaine, agrégé sur tous les dépôts)

In [ ]:
if not df.empty:
    # Agrégation sur tous les dépôts
    pivot = weekly.pivot_table(
        index="author", columns="week", values="pr_count", aggfunc="sum", fill_value=0
    )
    # Tri par nombre total de PR décroissant
    pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).index]
    pivot

## Tableau croisé par dépôt

In [ ]:
if not df.empty and len(REPOS) > 1:
    for repo_name, grp in weekly.groupby("repo"):
        pvt = grp.pivot_table(
            index="author", columns="week", values="pr_count", aggfunc="sum", fill_value=0
        )
        pvt = pvt.loc[pvt.sum(axis=1).sort_values(ascending=False).index]
        print(f"\n=== {repo_name} ===")
        display(pvt)

## Visualisation : nombre de PR fusionnées par semaine (empilé par auteur)

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(14, 5))

    stacked_height = None
    weeks = pivot.columns  # DatetimeIndex
    week_nums = mdates.date2num(weeks.to_pydatetime())

    for author in pivot.index:
        values = pivot.loc[author].values
        if stacked_height is None:
            ax.bar(week_nums, values, width=5, label=author)
            stacked_height = values.copy()
        else:
            ax.bar(week_nums, values, width=5, bottom=stacked_height, label=author)
            stacked_height += values

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=4))
    plt.xticks(rotation=45, ha="right")
    ax.set_xlabel("Semaine")
    ax.set_ylabel("Nombre de PR fusionnées")
    repos_label = ", ".join(f"{o}/{r}" for o, r in REPOS)
    ax.set_title(f"PR fusionnées par semaine — {repos_label}")
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1), title="Auteur")
    plt.tight_layout()
    plt.show()

## Visualisation : carte de chaleur (heatmap auteur × semaine)

In [ ]:
if not df.empty:
    fig, ax = plt.subplots(figsize=(14, max(3, len(pivot) * 0.5)))

    im = ax.imshow(pivot.values, aspect="auto", cmap="YlOrRd")
    plt.colorbar(im, ax=ax, label="Nombre de PR")

    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)

    # Affiche une étiquette de semaine sur 4
    step = max(1, len(pivot.columns) // 12)
    ax.set_xticks(range(0, len(pivot.columns), step))
    ax.set_xticklabels(
        [str(d)[:10] for d in pivot.columns[::step]], rotation=45, ha="right"
    )

    repos_label = ", ".join(f"{o}/{r}" for o, r in REPOS)
    ax.set_title(f"Heatmap des PR fusionnées — {repos_label}")
    ax.set_xlabel("Semaine")
    ax.set_ylabel("Auteur")
    plt.tight_layout()
    plt.show()